# Notebook 01: Chicago Crime Dataset Profiling (Pre-Cleaning)
## Scope: January 2024 to December 2025 Reported Incidents
**Author:** Data Engineering & Analytics Team

### Objective:
Thoroughly profile the raw Chicago Police Department crime dataset (City of Chicago Data Portal: `ijzp-q8t2`) prior to any data cleaning or imputation. Identify missing values, duplicate records, timestamp consistency, coordinate boundaries, and categorical code validity.

In [1]:
import os
import pandas as pd
import numpy as np

raw_path = '../data/raw/crimes_2024_2025.csv'
print(f'Loading raw data from: {raw_path}')
df_raw = pd.read_csv(raw_path, low_memory=False)
print(f'Dataset Dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

Loading raw data from: ../data/raw/crimes_2024_2025.csv


Dataset Dimensions: 497,919 rows x 21 columns


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude
0,27932,JH100028,2024-01-01T00:56:00.000,070XX S WABASH AVE,0110,HOMICIDE,FIRST DEGREE MURDER,HOUSE,False,False,...,3,6.0,69.0,01A,1177944.0,1858323.0,2024,2024-01-09T15:40:58.000,41.766555,-87.623324
1,27933,JH100028,2024-01-03T20:59:00.000,070XX S WABASH AVE,0110,HOMICIDE,FIRST DEGREE MURDER,HOUSE,False,False,...,3,6.0,69.0,01A,1177944.0,1858323.0,2024,2024-01-11T15:40:55.000,41.766555,-87.623324
2,27934,JH102838,2024-01-03T10:37:00.000,046XX S LARAMIE AVE,0110,HOMICIDE,FIRST DEGREE MURDER,HOUSE,True,False,...,8,22.0,56.0,01A,1142475.0,1873366.0,2024,2024-11-01T15:40:34.000,41.808566,-87.752960
3,27935,JH106979,2024-01-06T18:43:00.000,008XX W 72ND ST,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,...,7,17.0,68.0,01A,1171790.0,1857109.0,2024,2024-01-14T15:41:25.000,41.763361,-87.645916
4,27936,JH107122,2024-01-06T21:21:00.000,048XX W CHICAGO AVE,0110,HOMICIDE,FIRST DEGREE MURDER,RETAIL STORE,False,False,...,15,37.0,25.0,01A,1143907.0,1904871.0,2024,2024-01-14T15:41:25.000,41.894993,-87.746919


### 1. Attribute Data Types and Non-Null Summary

In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 497919 entries, 0 to 497918
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    497919 non-null  int64  
 1   case_number           497919 non-null  str    
 2   date                  497919 non-null  str    
 3   block                 497919 non-null  str    
 4   iucr                  497919 non-null  str    
 5   primary_type          497919 non-null  str    
 6   description           497919 non-null  str    
 7   location_description  495495 non-null  str    
 8   arrest                497919 non-null  bool   
 9   domestic              497919 non-null  bool   
 10  beat                  497919 non-null  int64  
 11  district              497919 non-null  int64  
 12  ward                  497918 non-null  float64
 13  community_area        497903 non-null  float64
 14  fbi_code              497919 non-null  str    
 15  x_coordinat

### 2. Missing Value Analysis

In [3]:
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100
missing_summary = pd.DataFrame({'Missing_Count': missing_counts, 'Missing_Pct': missing_pct.round(4)})
missing_summary[missing_summary['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

,Missing_Count,Missing_Pct
latitude,4904,0.9849
y_coordinate,4904,0.9849
x_coordinate,4904,0.9849
longitude,4904,0.9849
location_description,2424,0.4868
community_area,16,0.0032
ward,1,0.0002


### 3. Duplicate Identification: Exact vs. Logical Duplicates

In [4]:
exact_dups = df_raw.duplicated().sum()
id_dups = df_raw.duplicated(subset=['id']).sum()
case_dups = df_raw.duplicated(subset=['case_number']).sum()
print(f'Exact Row Duplicates: {exact_dups}')
print(f'Duplicate Incident Primary Keys (id): {id_dups}')
print(f'Records Sharing CPD Case Numbers (Logical Multi-Charge Incidents): {case_dups}')

Exact Row Duplicates: 0
Duplicate Incident Primary Keys (id): 0
Records Sharing CPD Case Numbers (Logical Multi-Charge Incidents): 51


### 4. Timestamp and Temporal Validity

In [5]:
df_raw['parsed_date'] = pd.to_datetime(df_raw['date'], errors='coerce')
unparseable = df_raw['parsed_date'].isnull().sum()
min_dt = df_raw['parsed_date'].min()
max_dt = df_raw['parsed_date'].max()
out_of_bounds_date = ((df_raw['parsed_date'] < '2024-01-01') | (df_raw['parsed_date'] > '2025-12-31 23:59:59')).sum()
print(f'Unparseable Timestamps: {unparseable}')
print(f'Earliest Reported Incident: {min_dt}')
print(f'Latest Reported Incident: {max_dt}')
print(f'Incidents Outside 2024-2025 Study Window: {out_of_bounds_date}')

Unparseable Timestamps: 0
Earliest Reported Incident: 2024-01-01 00:00:00
Latest Reported Incident: 2025-12-31 23:58:00
Incidents Outside 2024-2025 Study Window: 0


### 5. Geocoordinate Quality and Bounding Box Verification

In [6]:
coord_nulls = df_raw['latitude'].isnull().sum()
coord_zeroes = ((df_raw['latitude'] == 0) | (df_raw['longitude'] == 0)).sum()
valid_mask = df_raw['latitude'].notnull() & df_raw['longitude'].notnull()
out_of_bounds = valid_mask & ((df_raw['latitude'] < 41.60) | (df_raw['latitude'] > 42.06) | (df_raw['longitude'] < -87.95) | (df_raw['longitude'] > -87.50))
print(f'Missing Coordinates: {coord_nulls:,} ({coord_nulls/len(df_raw)*100:.2f}%)')
print(f'Coordinates at (0,0): {coord_zeroes}')
print(f'Valid Coordinates Outside Chicago Bounding Box: {out_of_bounds.sum()}')

Missing Coordinates: 4,904 (0.98%)
Coordinates at (0,0): 0
Valid Coordinates Outside Chicago Bounding Box: 0


### 6. Categorical Attributes & Offense Code Inspection

In [7]:
print(f'Unique Primary Crime Types: {df_raw["primary_type"].nunique()}')
print(f'Unique IUCR Offense Codes: {df_raw["iucr"].nunique()}')
print(f'Unique FBI Codes: {df_raw["fbi_code"].nunique()}')
print(f'Unique Location Descriptions: {df_raw["location_description"].nunique()}')
print(f'Arrest Flag Distribution:\n{df_raw["arrest"].value_counts(dropna=False)}')
print(f'Domestic Flag Distribution:\n{df_raw["domestic"].value_counts(dropna=False)}')

Unique Primary Crime Types: 31
Unique IUCR Offense Codes: 353
Unique FBI Codes: 26


Unique Location Descriptions: 134
Arrest Flag Distribution:
arrest
False    423540
True      74379
Name: count, dtype: int64
Domestic Flag Distribution:
domestic
False    404826
True      93093
Name: count, dtype: int64
